# Stage 13 — prepare separate deployment packages

This notebook prepares the **Python 3.10 FastAPI backend for Render** and the **Next.js frontend for Vercel** as two independent ZIP packages, ready to extract into separate Git repositories.

It builds the backend inference assets from Stage 05 and Stage 08 outputs, verifies the saved models, then packages each side separately. It does not read locked-test activities or predictions, refit models, or tune parameters.

**Run with the project's Python 3.10 environment.** Run cells in order. Review the asset and preflight results before pushing the backend package. The saved models are packaged with the backend ZIP; do not upload the local virtual environment.

In [1]:
# 1. Imports and project paths
from pathlib import Path
import sys
import json
import platform
import zipfile
import pandas as pd
import numpy as np
from IPython.display import display

# Find the BRD4 project root from either the root folder or notebooks/.
# This notebook prepares assets itself; prepare_stage13_assets.py is not required.
_search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    (p for p in _search_roots if (p / "config.json").is_file() and (p / "outputs").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the BRD4 project root. Start Jupyter from the folder containing config.json and outputs/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FRONTEND_SOURCE = PROJECT_ROOT / "stage13_webapp" / "frontend"
BACKEND_SOURCE = PROJECT_ROOT / "stage13_webapp" / "backend"
BACKEND_ASSETS = BACKEND_SOURCE / "app_assets"
PACKAGE_DIR = PROJECT_ROOT / "outputs" / "13_app_deployment" / "packages"
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {platform.python_version()}")
print(f"Frontend source: {FRONTEND_SOURCE}")
print(f"Backend source: {BACKEND_SOURCE}")
print(f"Separate ZIPs will be saved in: {PACKAGE_DIR}")
if platform.python_version_tuple()[:2] != ("3", "10"):
    print("WARNING: use the Python 3.10 environment.")

Project root: C:\Users\USER\Documents\E\Code
Python: 3.10.0
Frontend source: C:\Users\USER\Documents\E\Code\stage13_webapp\frontend
Backend source: C:\Users\USER\Documents\E\Code\stage13_webapp\backend
Separate ZIPs will be saved in: C:\Users\USER\Documents\E\Code\outputs\13_app_deployment\packages


## 2. Check required research outputs

The asset preparation uses only `config.json`, the fixed Stage 05 feature list and training feature matrix, and the three Stage 08 model bundles. It does not need a test-set file.

In [2]:
# 2. Confirm the required source files exist before preparing assets.
required_inputs = {
    "Stage 05 fixed feature names": PROJECT_ROOT / "outputs/05_feature_selection/tables/selected_features.csv",
    "Stage 05 training feature matrix": PROJECT_ROOT / "outputs/05_feature_selection/tables/train_selected_features.parquet",
    "Project config": PROJECT_ROOT / "config.json",
    "ExtraTrees model": PROJECT_ROOT / "outputs/08_final_model_evaluation/models/final_ExtraTreesRegressor.joblib",
    "CatBoost model": PROJECT_ROOT / "outputs/08_final_model_evaluation/models/final_CatBoostRegressor.joblib",
    "SVR model": PROJECT_ROOT / "outputs/08_final_model_evaluation/models/final_SVR.joblib",
}
input_status = pd.DataFrame([
    {"input": name, "path": str(file), "exists": file.is_file()}
    for name, file in required_inputs.items()
])
display(input_status)
missing_inputs = input_status.loc[~input_status["exists"], "path"].tolist()
if missing_inputs:
    raise FileNotFoundError("Complete Stages 05 and 08 first. Missing files:\n" + "\n".join(missing_inputs))

,input,path,exists
0,Stage 05 fixed feature names,C:\Users\USER\Documents\E\Code\outputs\05_feat...,True
1,Stage 05 training feature matrix,C:\Users\USER\Documents\E\Code\outputs\05_feat...,True
2,Project config,C:\Users\USER\Documents\E\Code\config.json,True
3,ExtraTrees model,C:\Users\USER\Documents\E\Code\outputs\08_fina...,True
4,CatBoost model,C:\Users\USER\Documents\E\Code\outputs\08_fina...,True
5,SVR model,C:\Users\USER\Documents\E\Code\outputs\08_fina...,True


## 3. Verify the fixed 250-feature list

The deployment uses the exact Stage 05 order: 103 chirality-aware Morgan bits, 146 Mordred Community 2D descriptors, and `phys_MolLogP`. No 3D descriptors or imputation are used.

In [3]:
# 3. Check feature count, composition, uniqueness, and finite training reference.
features_df = pd.read_csv(required_inputs["Stage 05 fixed feature names"])
if "feature" not in features_df:
    raise ValueError("selected_features.csv must contain a 'feature' column.")
selected_features = features_df["feature"].astype(str).tolist()
feature_counts = {
    "total": len(selected_features),
    "unique": len(set(selected_features)),
    "Morgan bits": sum(name.startswith("ecfp4_") for name in selected_features),
    "Mordred 2D": sum(name.startswith("mordred2d_") for name in selected_features),
    "phys_MolLogP": selected_features.count("phys_MolLogP"),
    "Mordred 3D": sum(name.startswith("mordred3d_") for name in selected_features),
}
display(pd.DataFrame([feature_counts]))
expected_counts = {"total": 250, "unique": 250, "Morgan bits": 103, "Mordred 2D": 146, "phys_MolLogP": 1, "Mordred 3D": 0}
if feature_counts != expected_counts:
    raise ValueError(f"Unexpected fixed-feature composition: {feature_counts}")

X_train_source = pd.read_parquet(required_inputs["Stage 05 training feature matrix"])
missing_columns = sorted(set(selected_features) - set(X_train_source.columns))
if missing_columns:
    raise ValueError(f"Stage 05 training matrix is missing selected features: {missing_columns[:10]}")
X_train_fixed = X_train_source.loc[:, selected_features].apply(pd.to_numeric, errors="coerce")
if X_train_fixed.empty or not np.isfinite(X_train_fixed.to_numpy(dtype=float)).all():
    raise ValueError("Training reference has missing/infinite values. This pipeline does not impute.")
print(f"Training reference rows: {len(X_train_fixed):,}; selected columns: {X_train_fixed.shape[1]}")

,total,unique,Morgan bits,Mordred 2D,phys_MolLogP,Mordred 3D
0,250,250,103,146,1,0


Training reference rows: 5,134; selected columns: 250


## 4. Generate backend assets

This creates `app_config.json` and `training_reference.joblib`, and copies the three saved model bundles into `stage13_webapp/backend/app_assets/`. The extra `final_model_bundle.joblib` name is not required by this app; it loads the three individual model files below.

In [4]:
# 4. Generate inference-only assets directly from completed research outputs.
# No helper script needs to be copied into the BRD4 project folder.
import shutil
import joblib

BACKEND_ASSETS.mkdir(parents=True, exist_ok=True)
selected_frame = pd.read_csv(required_inputs["Stage 05 fixed feature names"])
selected = selected_frame["feature"].astype(str).tolist()
cfg = json.loads(required_inputs["Project config"].read_text(encoding="utf-8"))

# Save the fixed-250 training reference used for applicability-domain calculations.
joblib.dump(
    {"X_train": X_train_fixed, "feature_names": selected},
    BACKEND_ASSETS / "training_reference.joblib",
    compress=3,
)
selected_frame.to_csv(BACKEND_ASSETS / "selected_features.csv", index=False)

# Verify model bundle format and Stage 05 feature order before copying.
for model_name in ("ExtraTreesRegressor", "CatBoostRegressor", "SVR"):
    source = PROJECT_ROOT / "outputs" / "08_final_model_evaluation" / "models" / f"final_{model_name}.joblib"
    bundle = joblib.load(source)
    if not isinstance(bundle, dict) or "model" not in bundle:
        raise ValueError(f"Unexpected saved bundle format for {model_name}.")
    if list(map(str, bundle.get("feature_names", []))) != selected:
        raise ValueError(f"{model_name} feature order differs from the fixed Stage 05 list.")
    shutil.copy2(source, BACKEND_ASSETS / f"final_{model_name}.joblib")

n_morgan = sum(name.startswith("ecfp4_") for name in selected)
n_mordred2d = sum(name.startswith("mordred2d_") for name in selected)
n_phys = sum(name.startswith("phys_") for name in selected)
n_mordred3d = sum(name.startswith("mordred3d_") for name in selected)
app_config = {
    "fingerprint_radius": int(cfg["fingerprint_radius"]),
    "fingerprint_bits": int(cfg["fingerprint_bits"]),
    "use_chirality": bool(cfg["use_chirality"]),
    "remove_salts": bool(cfg.get("remove_salts", True)),
    "feature_counts": {
        "Morgan": n_morgan,
        "Mordred 2D": n_mordred2d,
        "Physicochemical": n_phys,
        "Mordred 3D": n_mordred3d,
    },
    "n_training_reference_rows": int(len(X_train_fixed)),
    "n_features": len(selected),
    "ad_methods": ["standardized_5nn", "training_bounding_box", "leverage"],
}
(BACKEND_ASSETS / "app_config.json").write_text(
    json.dumps(app_config, indent=2), encoding="utf-8"
)
print(f"Prepared backend assets at: {BACKEND_ASSETS}")
print(f"Training reference rows: {len(X_train_fixed):,}; fixed features: {len(selected)}")
print("Locked-test data were not read.")

Prepared backend assets at: C:\Users\USER\Documents\E\Code\stage13_webapp\backend\app_assets
Training reference rows: 5,134; fixed features: 250
Locked-test data were not read.


In [5]:
# 5. Review the generated backend files and their sizes.
required_backend_assets = [
    "app_config.json",
    "selected_features.csv",
    "training_reference.joblib",
    "final_ExtraTreesRegressor.joblib",
    "final_CatBoostRegressor.joblib",
    "final_SVR.joblib",
]
asset_report = pd.DataFrame([
    {
        "file": name,
        "exists": (BACKEND_ASSETS / name).is_file(),
        "size_MB": round((BACKEND_ASSETS / name).stat().st_size / (1024 * 1024), 3)
            if (BACKEND_ASSETS / name).is_file() else None,
    }
    for name in required_backend_assets
])
display(asset_report)
missing_assets = asset_report.loc[~asset_report["exists"], "file"].tolist()
if missing_assets:
    raise FileNotFoundError("Backend assets were not generated: " + ", ".join(missing_assets))

,file,exists,size_MB
0,app_config.json,True,0.000
1,selected_features.csv,True,0.011
2,training_reference.joblib,True,2.882
3,final_ExtraTreesRegressor.joblib,True,144.506
4,final_CatBoostRegressor.joblib,True,1.672
5,final_SVR.joblib,True,7.423


## 5. Load and test the assets under Python 3.10

The preflight checks that all three saved estimators load, match the fixed feature order, and return a finite prediction for a training-reference row. Run this notebook in the same Python 3.10 environment used for the fitted models.

In [6]:
# 6. Check that all prepared backend models load and predict successfully.
import joblib
import numpy as np
import pandas as pd

required_assets = [
    BACKEND_ASSETS / "selected_features.csv",
    BACKEND_ASSETS / "training_reference.joblib",
    BACKEND_ASSETS / "final_ExtraTreesRegressor.joblib",
    BACKEND_ASSETS / "final_CatBoostRegressor.joblib",
    BACKEND_ASSETS / "final_SVR.joblib",
]
missing_assets = [path for path in required_assets if not path.is_file()]
if missing_assets:
    raise FileNotFoundError(
        "Run the asset-generation cell first. Missing files:\n"
        + "\n".join(map(str, missing_assets))
    )

selected = (
    pd.read_csv(BACKEND_ASSETS / "selected_features.csv")["feature"]
    .astype(str)
    .tolist()
)
reference = joblib.load(BACKEND_ASSETS / "training_reference.joblib")
X_reference = reference["X_train"]
saved_features = list(map(str, reference["feature_names"]))

if selected != saved_features:
    raise ValueError("Feature order in the training reference differs from selected_features.csv.")
if list(map(str, X_reference.columns)) != selected:
    raise ValueError("Training reference columns differ from the fixed feature list.")
if X_reference.empty or not np.isfinite(X_reference.to_numpy(dtype=float)).all():
    raise ValueError("Training reference contains missing or infinite values.")

# Use one training-reference row to confirm each saved model can make a finite prediction.
X_probe = X_reference.loc[:, selected].iloc[[0]]
preflight_rows = []

for model_name in ("ExtraTreesRegressor", "CatBoostRegressor", "SVR"):
    bundle = joblib.load(BACKEND_ASSETS / f"final_{model_name}.joblib")

    if not isinstance(bundle, dict) or "model" not in bundle:
        raise ValueError(f"Unexpected saved bundle format for {model_name}.")

    model_features = list(map(str, bundle.get("feature_names", [])))
    if model_features != selected:
        raise ValueError(f"{model_name} feature order differs from the fixed 250 features.")

    prediction = np.asarray(bundle["model"].predict(X_probe), dtype=float).reshape(-1)
    if prediction.size != 1 or not np.isfinite(prediction[0]):
        raise RuntimeError(f"{model_name} returned an invalid prediction: {prediction}")

    preflight_rows.append({
        "model": model_name,
        "status": "passed",
        "sample_predicted_pIC50": float(prediction[0]),
    })

display(pd.DataFrame(preflight_rows))
print(f"Preflight passed: {len(selected)} features; {len(X_reference):,} training reference molecules.")

,model,status,sample_predicted_pIC50
0,ExtraTreesRegressor,passed,6.249845
1,CatBoostRegressor,passed,6.237374
2,SVR,passed,6.159840


Preflight passed: 250 features; 5,134 training reference molecules.


## 6. Build two independent deployment ZIPs

The frontend ZIP contains only the Vercel Next.js app. The backend ZIP contains only the Render FastAPI app, Python 3.10 Docker configuration, dependencies, and the generated model/reference assets. Each ZIP extracts to its own repository root.

In [7]:
# 7. Package the frontend and backend separately for separate Git repositories.
def make_standalone_zip(source_dir: Path, zip_path: Path) -> Path:
    if not source_dir.is_dir():
        raise FileNotFoundError(f"Missing source folder: {source_dir}")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for file in source_dir.rglob("*"):
            if not file.is_file() or "__pycache__" in file.parts or file.suffix == ".pyc":
                continue
            archive.write(file, file.relative_to(source_dir))
    return zip_path

frontend_zip = make_standalone_zip(
    FRONTEND_SOURCE,
    PACKAGE_DIR / "BRD4_QSAR_Frontend_Standalone.zip",
)
backend_zip = make_standalone_zip(
    BACKEND_SOURCE,
    PACKAGE_DIR / "BRD4_QSAR_Backend_Standalone.zip",
)
package_report = pd.DataFrame([
    {"package": "Vercel frontend", "zip": str(frontend_zip), "size_MB": round(frontend_zip.stat().st_size / (1024 * 1024), 3)},
    {"package": "Render Python 3.10 backend", "zip": str(backend_zip), "size_MB": round(backend_zip.stat().st_size / (1024 * 1024), 3)},
])
display(package_report)

,package,zip,size_MB
0,Vercel frontend,C:\Users\USER\Documents\E\Code\outputs\13_app_...,0.008
1,Render Python 3.10 backend,C:\Users\USER\Documents\E\Code\outputs\13_app_...,54.039


## 7. Push and connect the two repositories

1. Extract `BRD4_QSAR_Frontend_Standalone.zip` into the frontend Git repository and deploy that repository on Vercel.
2. Extract `BRD4_QSAR_Backend_Standalone.zip` into a different backend Git repository and deploy it on Render as a Blueprint. The backend ZIP includes `render.yaml` and its Dockerfile.
3. In Vercel, set `NEXT_PUBLIC_API_URL` to the Render API base URL.
4. In Render, set `CORS_ORIGINS` to the exact Vercel frontend origin.
5. Check `https://<render-api-domain>/api/health`; it should list all three model names.

Render's free services can sleep when idle and take time to wake. Model assets are included in the backend ZIP after asset preparation, so keep that repository private if the model files or training reference should not be distributed publicly.

**Deployment is not triggered by this notebook.** It prepares assets and separate packages; you push each package to its destination repository and connect each service in its hosting dashboard.